
# Thí nghiệm 04 — Electricity: Huấn luyện và Grid Search

## 1. Nạp kiến trúc mô hình và khởi tạo môi trường

### 1.1. Mục tiêu

Notebook quản lý quy trình huấn luyện và đánh giá bốn mô hình dự báo chuỗi thời gian Electricity:

1. LSTM.
2. DeepKoopFormer-PatchTST.
3. DeepKoopFormer-Autoformer.
4. DeepKoopFormer-Informer.

Kiến trúc và các hàm dùng chung được định nghĩa trong `02_electricity_model.ipynb`.

### 1.2. Nạp định nghĩa bằng AST

Notebook sử dụng Abstract Syntax Tree (AST) để trích xuất các class và function từ Model notebook.

Phương pháp này chỉ nạp những định nghĩa cần thiết, tránh thực thi lại các cell khởi tạo dữ liệu hoặc những phép kiểm tra không liên quan.

Các định nghĩa bao gồm:

- Toán tử Koopman ổn định.
- Positional Encoding và Patch Embedding.
- Các backbone dựa trên Transformer.
- LSTM baseline.
- Các hàm tạo Sliding Windows, tính loss, huấn luyện và đánh giá.

### 1.3. Thiết lập môi trường

Thực nghiệm sử dụng:

- Random seed: 7.
- PyTorch.
- CUDA nếu khả dụng, ngược lại CPU.


### 1.4. Phạm vi thực nghiệm

Notebook định nghĩa quy trình huấn luyện, đánh giá và quản lý kết quả cho bốn kiến trúc dự báo chuỗi thời gian Electricity.

Các cấu hình thực nghiệm được tổ chức theo Patch Length, Forecast Horizon và loại mô hình.

Quá trình huấn luyện sử dụng dữ liệu đã được chuẩn bị từ notebook Data. Mỗi cấu hình tương ứng với một checkpoint chứa trọng số, lịch sử huấn luyện và kết quả đánh giá.

Các kết quả được lưu lại nhằm phục vụ phân tích sai số dự báo và so sánh giữa các kiến trúc.


Các thao tác huấn luyện chỉ được kích hoạt thông qua cơ chế điều khiển riêng ở cuối notebook.

In [ ]:
import ast
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

# 1. Reproducibility
SEED = 7

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Training execution control
RUN_TRAINING = False

# 2. Locate Model Notebook
candidates = [
    Path.cwd() / "02_electricity_model.ipynb",
    Path.cwd() / "experiments" / "04_electricity"
    / "02_electricity_model.ipynb",
    Path.cwd().parent / "experiments" / "04_electricity"
    / "02_electricity_model.ipynb",
]

MODEL_NOTEBOOK = next(
    (p.resolve() for p in candidates if p.is_file()),
    None
)

if MODEL_NOTEBOOK is None:
    raise FileNotFoundError(
        "Cannot find 02_electricity_model.ipynb"
    )

# 3. Read Notebook
with open(MODEL_NOTEBOOK, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# 4. Required Classes
required_classes = {
    "StrictStableKoopmanOperator",
    "SinCosPosEnc",
    "PatchEmbed1D",
    "PatchTST_Backbone",
    "Koopformer_PatchTST",
    "SeriesDecomp",
    "SimpleAutoformer",
    "Koopformer_Autoformer",
    "InformerSparse",
    "Koopformer_Informer",
    "SimpleLSTMForecaster",
}

# 5. Required Functions
required_functions = {
    "orthogonalize",
    "create_windows",
    "compute_loss",
    "train_model",
    "evaluate_model",
    "create_model",
}

# 6. Extract Definitions
definitions = []

for cell in notebook["cells"]:

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    # Parse Python source into AST
    tree = ast.parse(source)

    for node in tree.body:

        if (
            isinstance(node, ast.ClassDef)
            and node.name in required_classes
        ) or (
            isinstance(node, ast.FunctionDef)
            and node.name in required_functions
        ):
            definitions.append(node)

# 7. Compile and Load Definitions
module = ast.Module(
    body=definitions,
    type_ignores=[]
)

ast.fix_missing_locations(module)

exec(
    compile(
        module,
        str(MODEL_NOTEBOOK),
        "exec"
    ),
    globals()
)

# 8. Check Imported Definitions
required_names = required_classes | required_functions

missing = [
    name
    for name in required_names
    if name not in globals()
]

assert not missing, f"Missing definitions: {missing}"

print("=== ELECTRICITY TRAINING ===")

print("\nDevice:", DEVICE)
print("Model Notebook:", MODEL_NOTEBOOK)

print("\nImported classes:", len(required_classes))
print("Imported functions:", len(required_functions))

print("\nAll model definitions loaded successfully!")

=== ELECTRICITY TRAINING ===

Device: cuda
Model Notebook: C:\Users\Win 11 New\Desktop\pa gk\experiments\04_electricity\02_electricity_model.ipynb

Imported classes: 11
Imported functions: 6

All model definitions loaded successfully!



## 2. Dữ liệu đầu vào và cấu hình thực nghiệm

### 2.1. Bộ dữ liệu Electricity

Dữ liệu đã được chuẩn bị trong `01_electricity_data.ipynb`.

Tệp đầu vào:

`data/electricity_train_test.pt`

| Thành phần | Kích thước |
|---|---|
| Train | (2800, 6) |
| Test | (700, 6) |
| Số kênh | 6 |

Dữ liệu được chia theo thứ tự thời gian và chuẩn hóa bằng Min-Max Scaling, với scaler được fit trên tập Train.

### 2.2. Không gian Grid Search

| Tham số | Các giá trị |
|---|---|
| Patch Length | 70, 80, 90, 100, 110, 120, 130 |
| Forecast Horizon | 2, 4, 6, 8, 10, 12, 14, 16 |
| Models | LSTM, PatchTST, Autoformer, Informer |

Tổng số cấu hình:

$$
7\times8\times4=224
$$

### 2.3. Siêu tham số

| Tham số | Giá trị |
|---|---|
| Model dimension | 96 |
| LSTM hidden dimension | 48 |
| Epochs | 4000 |
| Learning rate | 0.0003 |
| Lyapunov weight | 0.1 |
| Optimizer | Adam |

Các mô hình Koopformer sử dụng chính quy hóa Lyapunov.

Đối với LSTM baseline, hệ số Lyapunov được đặt bằng 0 trong hàm huấn luyện từng cấu hình.

### 2.4. Tính nhất quán

Cell tiếp theo xác minh các thông số và tạo danh sách 224 cấu hình.

Việc khai báo cấu hình không thực hiện huấn luyện hoặc thay đổi các checkpoint đã lưu.


In [5]:
from itertools import product

# 1. Locate Electricity Dataset
DATA_PATH = (
    MODEL_NOTEBOOK.parent
    / "data"
    / "electricity_train_test.pt"
)

assert DATA_PATH.is_file(), f"Dataset not found: {DATA_PATH}"

# 2. Load Preprocessed Data
bundle = torch.load(
    DATA_PATH,
    map_location="cpu",
    weights_only=False
)

train_data = bundle["train"]
test_data = bundle["test"]

INPUT_DIM = train_data.shape[1]
CHANNELS = bundle["channels"]

# 3. Grid Search Parameters
PATCH_LENGTHS = [
    70, 80, 90, 100, 110, 120, 130
]

HORIZONS = [
    2, 4, 6, 8, 10, 12, 14, 16
]

MODELS = [
    "LSTM",
    "PatchTST",
    "Autoformer",
    "Informer"
]

# 4. Model Hyperparameters
D_MODEL = 96
LSTM_HIDDEN_DIM = 48

EPOCHS = 4000
LEARNING_RATE = 3e-4
LYAP_WEIGHT = 0.1

# 5. Generate All Configurations
CONFIGURATIONS = list(
    product(
        PATCH_LENGTHS,
        HORIZONS,
        MODELS
    )
)

TOTAL_RUNS = len(CONFIGURATIONS)

# 6. Validate Configuration
assert train_data.shape == (2800, 6)
assert test_data.shape == (700, 6)

assert INPUT_DIM == 6
assert len(CHANNELS) == INPUT_DIM

assert torch.isfinite(train_data).all()
assert torch.isfinite(test_data).all()

assert TOTAL_RUNS == 224

assert bundle["data_hash"] == (
    "c024e2bb9460bf6e35d6f3353b82b6"
    "d90bee422366be0d6b3bb769a340eb0f0f"
)

# 7. Print Experiment Configuration
print("=== ELECTRICITY GRID SEARCH ===")

print("\nTrain:", train_data.shape)
print("Test:", test_data.shape)
print("Input dimension:", INPUT_DIM)

print("\nModels:", MODELS)
print("Patch lengths:", PATCH_LENGTHS)
print("Horizons:", HORIZONS)

print("\nKoopman d_model:", D_MODEL)
print("LSTM hidden dimension:", LSTM_HIDDEN_DIM)

print("\nTotal configurations:", TOTAL_RUNS)
print("Epochs per configuration:", EPOCHS)

print("\nLearning rate:", LEARNING_RATE)
print("Lyapunov weight:", LYAP_WEIGHT)

print("\nDataset SHA256:", bundle["data_hash"])

# 8. Preview First Configurations
print("\nFirst 8 configurations:")

for i, (p, h, model_name) in enumerate(
    CONFIGURATIONS[:8],
    start=1
):
    print(
        f"{i:3d}. "
        f"Model={model_name:10s} | "
        f"P={p} | H={h}"
    )

print("\nGrid Search configuration validated!")

=== ELECTRICITY GRID SEARCH ===

Train: torch.Size([2800, 6])
Test: torch.Size([700, 6])
Input dimension: 6

Models: ['LSTM', 'PatchTST', 'Autoformer', 'Informer']
Patch lengths: [70, 80, 90, 100, 110, 120, 130]
Horizons: [2, 4, 6, 8, 10, 12, 14, 16]

Koopman d_model: 96
LSTM hidden dimension: 48

Total configurations: 224
Epochs per configuration: 4000

Learning rate: 0.0003
Lyapunov weight: 0.1

Dataset SHA256: c024e2bb9460bf6e35d6f3353b82b6d90bee422366be0d6b3bb769a340eb0f0f

First 8 configurations:
  1. Model=LSTM       | P=70 | H=2
  2. Model=PatchTST   | P=70 | H=2
  3. Model=Autoformer | P=70 | H=2
  4. Model=Informer   | P=70 | H=2
  5. Model=LSTM       | P=70 | H=4
  6. Model=PatchTST   | P=70 | H=4
  7. Model=Autoformer | P=70 | H=4
  8. Model=Informer   | P=70 | H=4

Grid Search configuration validated!



## 3. Huấn luyện một cấu hình và lưu checkpoint

### 3.1. Mục đích

Hàm `run_one_configuration()` thực hiện huấn luyện một mô hình tại một cặp Patch Length và Forecast Horizon xác định.

Đầu vào:

- `model_name`.
- `patch_len`.
- `horizon`.

### 3.2. Quy trình

1. Xác định đường dẫn checkpoint theo cấu hình.
2. Kiểm tra checkpoint đã tồn tại hay chưa.
3. Tạo Sliding Windows cho Train và Test.
4. Khởi tạo mô hình.
5. Xác định hệ số chính quy hóa Lyapunov.
6. Huấn luyện mô hình.
7. Đánh giá trên Train và Test.
8. Lưu checkpoint và kết quả.
9. Giải phóng các tensor và bộ nhớ GPU không còn sử dụng.

### 3.3. Hàm mất mát

Hàm mục tiêu:

$$
\mathcal{L}_{total}
=
\mathcal{L}_{MSE}
+
\lambda\mathcal{L}_{Lyap}
$$

Với:

- LSTM: $\lambda=0$.
- Koopformer: $\lambda=0.1$.

### 3.4. Nội dung checkpoint

Checkpoint lưu:

- `model_state_dict`: trọng số mô hình.
- `pred_train`, `truth_train`: dự báo và nhãn Train.
- `pred_test`, `truth_test`: dự báo và nhãn Test.
- `loss`, `mse_history`, `lyap_history`, `eig`: lịch sử huấn luyện.
- `train_mse`, `train_mae`, `test_mse`, `test_mae`: metric.
- Metadata: tên mô hình, Patch Length, Horizon, số epoch, các siêu tham số và thông tin dữ liệu.

### 3.5. Bảo toàn checkpoint

Nếu checkpoint đã tồn tại, hàm kiểm tra metadata và bỏ qua cấu hình đó.

Checkpoint mới được ghi qua tệp tạm `.pt.tmp` rồi đổi tên thành `.pt`.

Cơ chế này giúp tránh lưu trực tiếp một checkpoint chưa hoàn chỉnh vào đường dẫn đích.

Tuy nhiên, checkpoint chỉ lưu trọng số cuối cùng, không chứa trạng thái optimizer để tiếp tục huấn luyện chính xác từ một epoch bất kỳ.


### 3.6. Khả năng tiếp tục thực nghiệm

Trước khi huấn luyện một cấu hình, chương trình kiểm tra checkpoint tương ứng.

Nếu checkpoint đã tồn tại và metadata phù hợp với cấu hình yêu cầu, chương trình bỏ qua lượt huấn luyện đó.

Nếu checkpoint chưa tồn tại, chương trình thực hiện huấn luyện và lưu kết quả.

Cơ chế này cho phép triển khai Grid Search qua nhiều lần thực thi mà không cần huấn luyện lại các cấu hình đã hoàn thành.

Checkpoint lưu trọng số cuối cùng, không lưu trạng thái optimizer để tiếp tục chính xác từ một epoch trung gian.

In [6]:
# 1. Results Directory
RESULTS_DIR = (
    DATA_PATH.parent.parent
    / "results"
    / "electricity_grid_search_e4000"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# 2. Train One Configuration
def run_one_configuration(
    model_name,
    patch_len,
    horizon
):

    # Step 1: Checkpoint Path
    checkpoint_path = RESULTS_DIR / (
        f"{model_name.lower()}_p{patch_len}"
        f"_h{horizon}_d{D_MODEL}.pt"
    )

    # Skip completed configurations
    if checkpoint_path.exists():

        saved = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False
        )

        assert saved["data_hash"] == bundle["data_hash"]
        assert saved["epochs"] == EPOCHS
        assert saved["model"] == model_name
        assert saved["patch_len"] == patch_len
        assert saved["horizon"] == horizon
        assert saved["d_model"] == D_MODEL
        assert saved["input_dim"] == INPUT_DIM

        print(f"SKIP: {checkpoint_path.name}")

        return checkpoint_path

    print(
        f"\nSTART: {model_name} | "
        f"P={patch_len}, H={horizon}"
    )

    # Step 2: Sliding Windows
    X_train, y_train = create_windows(
        train_data,
        patch_len,
        horizon
    )

    X_test, y_test = create_windows(
        test_data,
        patch_len,
        horizon
    )

    print("X_train:", X_train.shape)
    print("y_train:", y_train.shape)

    print("X_test:", X_test.shape)
    print("y_test:", y_test.shape)

    # Step 3: Initialize Model
    model = create_model(
        model_name=model_name,
        patch_len=patch_len,
        horizon=horizon
    )

    # Step 4: Lyapunov Weight
    lyap_weight = (
        0.0 if model_name == "LSTM"
        else LYAP_WEIGHT
    )

    # Step 5: Training
    result = train_model(
        model=model,
        X_train=X_train,
        y_train=y_train,
        epochs=EPOCHS,
        lr=LEARNING_RATE,
        lyap_weight=lyap_weight,
        verbose=True
    )

    trained_model = result["model"]

    # Step 6: Evaluation
    train_metrics = evaluate_model(
        trained_model,
        X_train,
        y_train
    )

    test_metrics = evaluate_model(
        trained_model,
        X_test,
        y_test
    )

    # Step 7: Build Checkpoint
    checkpoint = {

        # Model Information
        "model": model_name,

        "model_state_dict": {
            k: v.detach().cpu().clone()
            for k, v in trained_model.state_dict().items()
        },

        # Train Predictions
        "pred_train": train_metrics["predictions"],
        "truth_train": y_train.cpu(),

        # Test Predictions
        "pred_test": test_metrics["predictions"],
        "truth_test": y_test.cpu(),

        # Training Histories
        "loss": torch.tensor(result["loss"]),
        "eig": torch.tensor(result["eig"]),
        "mse_history": torch.tensor(result["mse"]),
        "lyap_history": torch.tensor(result["lyap"]),

        # Train Metrics
        "train_mse": train_metrics["MSE"],
        "train_mae": train_metrics["MAE"],

        # Test Metrics
        "test_mse": test_metrics["MSE"],
        "test_mae": test_metrics["MAE"],

        # Hyperparameters
        "patch_len": patch_len,
        "horizon": horizon,
        "d_model": D_MODEL,
        "lstm_hidden_dim": LSTM_HIDDEN_DIM,
        "input_dim": INPUT_DIM,

        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "lyap_weight": lyap_weight,

        # Dataset Metadata
        "channels": CHANNELS,
        "data_hash": bundle["data_hash"],
    }

    # Step 8: Atomic Checkpoint Save
    temp_path = checkpoint_path.with_suffix(
        ".pt.tmp"
    )

    torch.save(checkpoint, temp_path)

    temp_path.replace(checkpoint_path)

    # Step 9: Print Results
    print(
        f"\nDONE: {model_name} | "
        f"P={patch_len}, H={horizon}"
    )

    print(
        f"Train MSE: {train_metrics['MSE']:.6f} | "
        f"Train MAE: {train_metrics['MAE']:.6f}"
    )

    print(
        f"Test MSE: {test_metrics['MSE']:.6f} | "
        f"Test MAE: {test_metrics['MAE']:.6f}"
    )

    print("Saved:", checkpoint_path)

    # Step 10: Release Memory
    del trained_model, model, result
    del X_train, y_train, X_test, y_test

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return checkpoint_path


print("Electricity training function defined successfully.")
print("Results directory:", RESULTS_DIR)

Electricity training function defined successfully.
Results directory: C:\Users\Win 11 New\Desktop\pa gk\experiments\04_electricity\results\electricity_grid_search_e4000



## 4. Grid Search và quản lý kết quả

### 4.1. Mục tiêu

Grid Search đánh giá bốn mô hình trên 56 cặp Patch Length và Forecast Horizon, tương ứng tổng cộng 224 cấu hình.

Mỗi cấu hình được liên kết với một checkpoint riêng.

### 4.2. Quy ước checkpoint

Các checkpoint được lưu tại:

`results/electricity_grid_search_e4000/`

Quy tắc đặt tên:

`{model}_p{patch_len}_h{horizon}_d96.pt`

### 4.3. Hàm checkpoint_file()

Hàm xây dựng đường dẫn checkpoint từ tên mô hình và các tham số của cấu hình.

### 4.4. Hàm collect_metrics()

Hàm đọc các checkpoint đã hoàn thành, xác minh metadata và tổng hợp hai dòng metric cho mỗi checkpoint:

- Train.
- Test.

Các metric gồm MSE và MAE.

Kết quả được lưu vào `metrics.csv`.

**Lưu ý:** Hàm này có thao tác ghi lại `metrics.csv`. Vì vậy không gọi `collect_metrics()` trong quá trình kiểm toán chỉ đọc.

### 4.5. Hàm run_grid_search()

Hàm thực hiện:

1. Xác định các cấu hình đã hoàn thành.
2. Bỏ qua checkpoint đã tồn tại.
3. Huấn luyện các cấu hình còn thiếu.
4. Cập nhật bảng metric sau mỗi lượt huấn luyện.
5. Tổng hợp trạng thái Grid Search.

Tham số `max_new_runs` giới hạn số cấu hình mới được huấn luyện trong một lần chạy.


### 4.6. Tổ chức kết quả thực nghiệm

Mỗi cấu hình Grid Search được lưu dưới dạng checkpoint riêng biệt.

Hàm `collect_metrics()` tổng hợp kết quả từ các checkpoint thành bảng metric chung, bao gồm sai số trên tập Train và Test.

Hàm `run_grid_search()` quản lý toàn bộ quá trình duyệt cấu hình, kiểm tra trạng thái hoàn thành, huấn luyện và cập nhật kết quả.

Cách tổ chức này cho phép theo dõi tiến độ thực nghiệm và truy xuất kết quả của từng mô hình theo cấu hình tương ứng.

In [7]:
import os
from itertools import product

# 1. Metrics Configuration
METRICS_PATH = RESULTS_DIR / "metrics.csv"

METRIC_COLUMNS = [
    "Model",
    "PatchLen",
    "Horizon",
    "d_model",
    "Set",
    "Epochs",
    "MSE",
    "MAE"
]

CONFIGURATIONS = list(
    product(PATCH_LENGTHS, HORIZONS, MODELS)
)

assert len(CONFIGURATIONS) == 224


# 2. Checkpoint Path
def checkpoint_file(model_name, patch_len, horizon):

    return RESULTS_DIR / (
        f"{model_name.lower()}_p{patch_len}"
        f"_h{horizon}_d{D_MODEL}.pt"
    )


# 3. Collect Metrics from Checkpoints
def collect_metrics():

    rows = []
    completed = 0

    for patch_len, horizon, model_name in CONFIGURATIONS:

        path = checkpoint_file(
            model_name,
            patch_len,
            horizon
        )

        # Bỏ qua cấu hình chưa huấn luyện
        if not path.exists():
            continue

        # Load checkpoint
        ckpt = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

        # Kiểm tra checkpoint thuộc đúng thực nghiệm
        assert ckpt["data_hash"] == bundle["data_hash"]
        assert ckpt["epochs"] == EPOCHS
        assert ckpt["model"] == model_name
        assert ckpt["patch_len"] == patch_len
        assert ckpt["horizon"] == horizon
        assert ckpt["d_model"] == D_MODEL
        assert ckpt["input_dim"] == INPUT_DIM
        assert ckpt["lstm_hidden_dim"] == LSTM_HIDDEN_DIM

        completed += 1

        # Mỗi checkpoint có hai bộ metrics: Train và Test
        for dataset in ["Train", "Test"]:

            prefix = dataset.lower()

            rows.append({
                "Model": model_name,
                "PatchLen": patch_len,
                "Horizon": horizon,
                "d_model": D_MODEL,
                "Set": dataset,
                "Epochs": EPOCHS,
                "MSE": ckpt[f"{prefix}_mse"],
                "MAE": ckpt[f"{prefix}_mae"]
            })

    # Tạo DataFrame
    metrics_df = pd.DataFrame(
        rows,
        columns=METRIC_COLUMNS
    )

    # Lưu metrics.csv an toàn
    temp_path = METRICS_PATH.with_suffix(
        ".csv.tmp"
    )

    metrics_df.to_csv(
        temp_path,
        index=False
    )

    os.replace(
        temp_path,
        METRICS_PATH
    )

    return completed, metrics_df


# 4. Grid Search Function
def run_grid_search(max_new_runs=None):

    # Kiểm tra những cấu hình đã hoàn thành
    completed, _ = collect_metrics()

    print("=== ELECTRICITY GRID SEARCH ===")
    print("Total configurations:", len(CONFIGURATIONS))
    print("Already completed:", completed)
    print("Epochs per configuration:", EPOCHS)
    print("Device:", DEVICE)

    new_runs = 0

    # Duyệt toàn bộ 224 cấu hình
    for index, (patch_len, horizon, model_name) in enumerate(
        CONFIGURATIONS,
        start=1
    ):

        path = checkpoint_file(
            model_name,
            patch_len,
            horizon
        )

        # Skip nếu đã hoàn thành
        if path.exists():

            print(
                f"SKIP [{index}/224] {path.name}"
            )

            continue

        # Giới hạn số lượt huấn luyện mới
        if (
            max_new_runs is not None
            and new_runs >= max_new_runs
        ):
            break

        print(
            f"\n[{index}/224] "
            f"{model_name} | "
            f"P={patch_len}, H={horizon}"
        )

        # Huấn luyện một cấu hình
        run_one_configuration(
            model_name=model_name,
            patch_len=patch_len,
            horizon=horizon
        )

        new_runs += 1

        # Cập nhật metrics sau mỗi lượt huấn luyện
        completed, metrics_df = collect_metrics()

        print(
            f"PROGRESS: {completed}/224 | "
            f"Metrics rows: {len(metrics_df)}"
        )

    # Tổng hợp kết quả cuối
    completed, metrics_df = collect_metrics()

    print("\n=== GRID SEARCH STATUS ===")
    print("Completed:", completed, "/224")
    print("New runs:", new_runs)
    print("Metrics rows:", len(metrics_df))
    print("Results directory:", RESULTS_DIR)

    return metrics_df


# 5. Check Definitions
print("Grid Search functions defined successfully.")
print("Total configurations:", len(CONFIGURATIONS))
print("Metrics path:", METRICS_PATH)

Grid Search functions defined successfully.
Total configurations: 224
Metrics path: C:\Users\Win 11 New\Desktop\pa gk\experiments\04_electricity\results\electricity_grid_search_e4000\metrics.csv



## 5. Kiểm toán kết quả huấn luyện đã lưu

### 5.1. Mục tiêu

Kiểm tra trạng thái hoàn thành của Grid Search Electricity từ những checkpoint và metric đã lưu.

Bước kiểm toán chỉ đọc dữ liệu, không huấn luyện hoặc cập nhật kết quả.

### 5.2. Các điều kiện kiểm tra

Grid Search dự kiến gồm:

- 7 Patch Length.
- 8 Forecast Horizon.
- 4 mô hình.
- Tổng cộng 224 checkpoint.

Mỗi checkpoint hoàn chỉnh đóng góp hai dòng metric, một cho Train và một cho Test.

Nếu toàn bộ cấu hình đã hoàn thành, số dòng dự kiến trong `metrics.csv` là:

$$
224\times2=448
$$

### 5.3. Kiểm tra metadata

Đối với checkpoint hiện có, kiểm tra:

- Tên mô hình.
- Patch Length và Forecast Horizon.
- Model dimension và Input dimension.
- Số epoch.
- Fingerprint SHA256 của dữ liệu.
- Sự tồn tại của trọng số và metric Train/Test.


### 5.4. Kết quả kiểm toán

Kết quả kiểm toán bao gồm:

- Tổng số cấu hình dự kiến.
- Số checkpoint đã tồn tại.
- Số checkpoint còn thiếu.
- Số checkpoint không đáp ứng điều kiện kiểm tra.
- Số bản ghi metric hiện có.

Những thông tin này được sử dụng để xác định mức độ hoàn thành và tính nhất quán của các kết quả thực nghiệm.

In [ ]:
# 1. Read-only audit
assert RUN_TRAINING is False, (
    "Disable training before running the audit."
)

expected_paths = [
    checkpoint_file(model_name, p, h)
    for p in PATCH_LENGTHS
    for h in HORIZONS
    for model_name in MODELS
]

existing_paths = [
    path for path in expected_paths
    if path.is_file()
]

missing_paths = [
    path for path in expected_paths
    if not path.is_file()
]

assert len(expected_paths) == 224

# 2. Validate existing checkpoint metadata
invalid_checkpoints = []

for path in existing_paths:
    try:
        ckpt = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

        assert ckpt["data_hash"] == bundle["data_hash"]
        assert ckpt["epochs"] == EPOCHS
        assert ckpt["d_model"] == D_MODEL
        assert ckpt["input_dim"] == INPUT_DIM

        assert "model_state_dict" in ckpt

        for key in (
            "train_mse", "train_mae",
            "test_mse", "test_mae"
        ):
            assert key in ckpt

    except (AssertionError, KeyError, RuntimeError,
            ValueError, OSError) as exc:
        invalid_checkpoints.append(
            (path.name, str(exc))
        )

# 3. Read existing metrics without modifying it
if METRICS_PATH.is_file():
    metrics_df = pd.read_csv(METRICS_PATH)
    metric_rows = len(metrics_df)
else:
    metric_rows = None

# 4. Summary
print("=== ELECTRICITY RESULTS AUDIT ===")
print("Expected checkpoints:", len(expected_paths))
print("Existing checkpoints:", len(existing_paths))
print("Missing checkpoints:", len(missing_paths))
print("Invalid checkpoints:", len(invalid_checkpoints))
print("Metrics rows:", metric_rows)
print("Expected metric rows:", 448)

if missing_paths:
    print("\nFirst missing checkpoints:")
    for path in missing_paths[:10]:
        print("-", path.name)

if invalid_checkpoints:
    print("\nInvalid checkpoints:")
    for name, reason in invalid_checkpoints[:10]:
        print("-", name, reason)

print("\nRead-only audit completed.")
print("No files were modified.")


## 6. Điều khiển thực thi Grid Search

### 6.1. Mục đích

Hàm `run_electricity_experiment()` cung cấp điểm vào để điều khiển quá trình Grid Search trực tiếp từ Jupyter Notebook.

Hàm sử dụng lại những thành phần đã định nghĩa:

- `create_model()`.
- `run_one_configuration()`.
- `checkpoint_file()`.
- `collect_metrics()`.
- `run_grid_search()`.

### 6.2. Cơ chế thực thi

Trước khi huấn luyện, hàm xác định số checkpoint đã tồn tại và số cấu hình còn thiếu.

Nếu toàn bộ checkpoint đã có, hàm chỉ đọc `metrics.csv` và không huấn luyện.

Nếu còn cấu hình chưa hoàn thành, việc huấn luyện chỉ được kích hoạt khi:

- `RUN_TRAINING = True`.
- `confirm = True`.

### 6.3. Giới hạn số lượt huấn luyện

Tham số `max_new_runs` xác định số cấu hình mới tối đa được huấn luyện trong một lần gọi.

Mặc định là 1, tương ứng với cách chạy giới hạn một cấu hình của script trước đây.


In [ ]:
def run_electricity_experiment(
    max_new_runs=1,
    confirm=False
):
    # 1. Validate arguments
    if not isinstance(max_new_runs, int) or max_new_runs < 1:
        raise ValueError(
            "max_new_runs must be a positive integer."
        )

    # 2. Check existing checkpoints
    expected_paths = [
        checkpoint_file(model, p, h)
        for p in PATCH_LENGTHS
        for h in HORIZONS
        for model in MODELS
    ]

    completed_paths = [
        path for path in expected_paths
        if path.is_file()
    ]

    missing_paths = [
        path for path in expected_paths
        if not path.is_file()
    ]

    print("=== ELECTRICITY GRID SEARCH ===")
    print("Device:", DEVICE)
    print("Total configurations:", len(expected_paths))
    print("Completed:", len(completed_paths))
    print("Missing:", len(missing_paths))
    print("Epochs:", EPOCHS)
    print("Max new runs:", max_new_runs)
    print("Data hash:", bundle["data_hash"])

    # 3. All checkpoints already exist
    if not missing_paths:
        print("\nAll checkpoints already exist.")
        print("No training required.")

        if not METRICS_PATH.is_file():
            raise FileNotFoundError(METRICS_PATH)

        return pd.read_csv(METRICS_PATH)

    # 4. Require explicit authorization
    if not RUN_TRAINING or not confirm:
        print("\nTraining disabled.")
        print("Existing checkpoints remain unchanged.")
        return None

    # 5. Execute only when enabled and confirmed
    print("\nStarting Electricity Grid Search...")

    metrics_df = run_grid_search(
        max_new_runs=max_new_runs
    )

    print("\nGrid Search execution finished.")
    return metrics_df


print("Electricity Grid Search entry point ready.")
print("Training enabled:", RUN_TRAINING)

## 7. Điều khiển chế độ thực thi

### 7.1. Chế độ mặc định

Notebook được đặt ở chế độ không huấn luyện:

`RUN_TRAINING = False`

Trạng thái này bảo vệ các checkpoint và metric hiện có trong quá trình kiểm tra, chỉnh sửa Markdown hoặc refactor.

### 7.2. Tiếp tục Grid Search

Khi có nhu cầu huấn luyện các cấu hình chưa hoàn thành, người dùng có thể chủ động bật chế độ huấn luyện và đặt giới hạn số lượt chạy.

Mặc định:

`MAX_NEW_RUNS = 1`

### 7.3. Lưu ý

Không bật chế độ huấn luyện trong quá trình dọn dẹp repository.

Cell này không thực hiện huấn luyện khi `RUN_TRAINING = False`.


In [ ]:
# Keep False during repository cleanup
RUN_TRAINING = False

# Maximum number of new training configurations
MAX_NEW_RUNS = 1


if RUN_TRAINING:
    electricity_metrics = run_electricity_experiment(
        max_new_runs=MAX_NEW_RUNS,
        confirm=True
    )
else:
    print("Training is disabled.")
    print("No training or checkpoint modifications.")